In [ ]:
# --------------------------------
# Imports & Path Setup
# --------------------------------

%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

import matplotlib.pyplot as plt
from PIL import Image
import torch

from src.compression.fourier import FourierCompressor
from src.utils.preprocessing import CompressionDataset, get_dataloaders

In [ ]:
# --------------------------------
# Parameters
# --------------------------------

RAW_DIR = Path("../data/raw")
RESIZED_DIR = Path("../data/resized")
COMP_DIR = Path("../data/compressed")

IMAGE_SIZE = 256
BATCH_SIZE = 8
SPLITS = (0.70, 0.15, 0.15)
SEED = 42

In [ ]:
# --------------------------------
# Resize & Save
# --------------------------------

RESIZED_DIR.mkdir(exist_ok=True)

raw_files = sorted(RAW_DIR.glob("*"))
raw_files = [f for f in raw_files if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]

print(f"Images found: {len(raw_files)}")

for path in raw_files:
    img = Image.open(path).convert("RGB")
    resized = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
    out = RESIZED_DIR / (path.stem + ".png")
    resized.save(out)

print(f"Resized images saved in: {RESIZED_DIR}")

Images found: 100
✓ Resized images saved in: ../data/resized


In [ ]:
# --------------------------------
# Compress with Fourier & Save
# --------------------------------

COMP_DIR.mkdir(exist_ok=True)
fourier = FourierCompressor(keep_fraction=0.25)

resized_files = sorted(RESIZED_DIR.glob("*.png"))
print(f"Images to compress: {len(resized_files)}")

for input_path in RESIZED_DIR.glob("*.png"):
    fourier.compress(input_image_path=str(input_path), output_dir=str(COMP_DIR))
    print(f"Processed: {input_path.name}")

print(f"Compressed images saved in: {COMP_DIR}")

Images to compress: 100
Processed: 117025.png
Processed: 176051.png
Processed: 223004.png
Processed: 226033.png
Processed: 15011.png
Processed: 147077.png
Processed: 107014.png
Processed: 20069.png
Processed: 104055.png
Processed: 2018.png
Processed: 196088.png
Processed: 181021.png
Processed: 225022.png
Processed: 189096.png
Processed: 217090.png
Processed: 103078.png
Processed: 183066.png
Processed: 108069.png
Processed: 14085.png
Processed: 207049.png
Processed: 189029.png
Processed: 10081.png
Processed: 198087.png
Processed: 226022.png
Processed: 17067.png
Processed: 118015.png
Processed: 102062.png
Processed: 160006.png
Processed: 209021.png
Processed: 157032.png
Processed: 112056.png
Processed: 147080.png
Processed: 196027.png
Processed: 187099.png
Processed: 120003.png
Processed: 159002.png
Processed: 109055.png
Processed: 101084.png
Processed: 134049.png
Processed: 106047.png
Processed: 196040.png
Processed: 108004.png
Processed: 120093.png
Processed: 146074.png
Processed: 1450

In [ ]:
# --------------------------------
# Sanity Check: shape consistency
# --------------------------------

errors = []
for comp_path in sorted(COMP_DIR.glob("*.png")):
    raw_id = comp_path.stem.replace("_fourier_25", "")
    raw_path = RESIZED_DIR / (raw_id + ".png")

    if not raw_path.exists():
        errors.append(f"Missing raw: {raw_id}")
        continue

    comp = Image.open(comp_path)
    orig = Image.open(raw_path)

    if comp.size != orig.size:
        errors.append(f"Size mismatch: {raw_id}  {comp.size} vs {orig.size}")

if errors:
    for e in errors:
        print(f"✗ {e}")
else:
    print(f" All {len(list(COMP_DIR.glob('*.png')))} pairs are consistent")

✓ All 100 pairs are consistent


In [ ]:
# --------------------------------
# Dataset & DataLoaders
# --------------------------------

dataset = CompressionDataset(
    raw_dir=RESIZED_DIR,
    compressed_dir=COMP_DIR,
    image_size=IMAGE_SIZE,
)

train_loader, val_loader, test_loader = get_dataloaders(
    dataset,
    splits=SPLITS,
    batch_size=BATCH_SIZE,
    seed=SEED,
    num_workers=0,
)

print(f"Total : {len(dataset)}")
print(f"Train : {len(train_loader.dataset)}")
print(f"Val : {len(val_loader.dataset)}")
print(f"Test : {len(test_loader.dataset)}")

Total  : 100
Train  : 70
Val    : 15
Test   : 15


In [ ]:
# --------------------------------
# Tensor Shape Check
# --------------------------------

comp, orig = dataset[0]

print(f"Shape : {comp.shape}")
print(f"Dtype : {comp.dtype}")
print(f"Range : [{comp.min():.3f}, {comp.max():.3f}]")

assert comp.shape == orig.shape
assert comp.shape == torch.Size([3, IMAGE_SIZE, IMAGE_SIZE])
print("Shape check passed")

Shape : torch.Size([3, 256, 256])
Dtype : torch.float32
Range : [0.075, 0.992]
✓ Shape check passed
